# 01 · Mise en place et exploration des données

**Missions couvertes :** 2 (cache) ✅ · 3 (charger une session) ✅ · 4 (explorer les tours) ✏️

**Objectif :** préparer l'environnement, charger une session de Formule 1 avec [FastF1](https://docs.fastf1.dev/)
et comprendre la structure des données avant toute analyse.

**Prérequis**
- L'environnement virtuel `.venv` est activé et les dépendances de `requirements.txt` sont installées.
- Jupyter Lab est lancé depuis la racine du repo (le notebook, lui, s'exécute depuis `notebooks/`).
- Pour les données de tours et la télémétrie, un compte Formula 1 est nécessaire. Connexion à faire une seule fois dans un terminal :
  `python -m fastf1 auth f1tv --authenticate`

**Légende :** ✅ mission terminée · ✏️ cellule à compléter par toi

## Imports

| Bibliothèque | Rôle dans le projet |
|---|---|
| `os` | manipuler les chemins de fichiers et les dossiers |
| `fastf1` | télécharger les données officielles de Formule 1 |
| `numpy` | calcul numérique (statistiques, conversions) |
| `pandas` | manipuler des tableaux de données (*DataFrame*) |
| `matplotlib.pyplot` | tracer des graphiques |
| `display` | afficher un tableau proprement dans Jupyter (au lieu de `print`) |

In [1]:
import os

import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## Mission 2 ✅ · Activer le cache FastF1

**Pourquoi un cache ?** Chaque session pèse plusieurs dizaines de Mo. Sans cache, FastF1 les retéléchargerait
à chaque exécution. Avec le cache, le premier chargement est lent et les suivants sont presque instantanés.
FastF1 range les fichiers par année, Grand Prix et session (`cache/2024/2024-05-26_Monaco_Grand_Prix/...`).

**Ce que fait la cellule, ligne par ligne :**
1. `assert ...` : le notebook s'exécute depuis `notebooks/`, donc `../cache` désigne le dossier `cache/` à la racine du repo.
   On vérifie ce point **avant** de créer quoi que ce soit, pour ne jamais créer un `cache/` au mauvais endroit.
   Si la condition est fausse, l'exécution s'arrête avec un message clair.
2. `os.path.abspath(...)` : convertit le chemin relatif en chemin absolu, une seule fois, dans une variable unique.
3. `os.makedirs(..., exist_ok=True)` : crée le dossier s'il n'existe pas, sans erreur s'il existe déjà.
   C'est nécessaire, car FastF1 refuse un dossier de cache inexistant.
4. `print(...)` : permet de vérifier visuellement que le chemin finit bien par `f1-telemetry-analysis/cache`.
5. `fastf1.Cache.configure(cache_dir=...)` : active le cache. Depuis FastF1 3.9, cette fonction remplace `enable_cache`
   (dépréciée). Le paramètre doit être **nommé** (`cache_dir=`), car il est déclaré après un `*` dans la définition de la fonction.

Le dossier `cache/` est listé dans le `.gitignore` : il ne part jamais sur GitHub.

In [2]:
# Le chemin relatif "../cache" n'est correct que si le notebook tourne depuis notebooks/
assert os.path.basename(os.getcwd()) == "notebooks", f"Dossier courant inattendu : {os.getcwd()}"

cache_dir = os.path.abspath("../cache")
os.makedirs(cache_dir, exist_ok=True)
print("Cache FastF1 :", cache_dir)

fastf1.Cache.configure(cache_dir=cache_dir)

Cache FastF1 : /home/olyex/Projects/f1-telemetry-analysis/cache


## Fonction utilitaire · `afficher`

`print(df)` transforme un tableau en texte brut : les colonnes sont coupées par des `...` et les retours à la ligne
rendent le tout illisible. Cette fonction affiche un vrai tableau Jupyter, avec deux options :

```python
afficher(df)                      # tout le tableau
afficher(df, n=5)                 # les 5 premières lignes
afficher(df, ["col_a", "col_b"])  # seulement ces colonnes
```

`pd.option_context("display.max_columns", None)` demande à pandas de ne masquer aucune colonne,
uniquement le temps de cet affichage (grâce au `with`).

In [3]:
def afficher(df, colonnes=None, n=None):
    '''Affiche un DataFrame sous forme de tableau Jupyter, sans colonnes tronquées.

    colonnes : liste de noms de colonnes à garder (toutes par défaut)
    n        : nombre de lignes à afficher (toutes par défaut)
    '''
    if colonnes is not None:
        df = df[colonnes]
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None):
        display(df)

## Mission 3 ✅ · Charger une session

`fastf1.get_session(annee, grand_prix, type_session)` attend trois informations :
- **l'année** : `2024` ;
- **le Grand Prix** : un nom (`"Monaco"`) ou un numéro de manche (`8`) ;
- **le type de session** : `"FP1"`, `"FP2"`, `"FP3"`, `"Q"` (qualifications), `"S"` (sprint), `"R"` (course).

`get_session` ne télécharge rien : il prépare seulement un objet `Session`. C'est `.load()` qui télécharge
(ou relit depuis le cache). On sépare les deux étapes pour pouvoir choisir quoi charger :
`session.load(telemetry=False)` par exemple, si on n'a pas besoin de la télémétrie.

Les paramètres sont mis dans des **constantes** en majuscules en haut de cellule : pour analyser une autre
course, il suffit de changer ces trois lignes.

In [4]:
ANNEE = 2024
GRAND_PRIX = "Monaco"
TYPE_SESSION = "Q"

session = fastf1.get_session(ANNEE, GRAND_PRIX, TYPE_SESSION)
session.load()
print(session.event["EventName"], "-", session.name)

core           INFO 	Loading data for Monaco Grand Prix - Qualifying [v3.9.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '55', '4', '63', '1', '44', '22', '23', '10', '31', '3', '18', '27', '14', '2', '20', '11', '77', '24']


Monaco Grand Prix - Qualifying


### Vérifier que les tours sont bien chargés

Sans connexion au compte F1, FastF1 récupère les résultats (via une autre source, Ergast)
mais **pas** les tours ni la télémétrie : accéder à `session.laps` provoque alors une `DataNotLoadedError`.
Cette cellule intercepte l'erreur avec `try` / `except` et affiche un message clair à la place.

Le message `Failed to authenticate with Formula 1 account` dans les logs signifie la même chose.

In [5]:
# Sans connexion au compte F1, FastF1 ne récupère que les résultats :
# accéder à session.laps lève alors une DataNotLoadedError.
try:
    nb_tours = len(session.laps)
except fastf1.exceptions.DataNotLoadedError:
    nb_tours = 0

if nb_tours == 0:
    print("⚠️ Aucun tour chargé. Connecte-toi dans un terminal avec :")
    print("   python -m fastf1 auth f1tv --authenticate")
else:
    print(f"✅ {nb_tours} tours chargés")

✅ 426 tours chargés


### Résultats de la session

`session.results` est un DataFrame avec une ligne par pilote : identité, équipe, position, temps de Q1/Q2/Q3, etc.
Les pilotes éliminés en Q1 ou en Q2 ont `NaT` (*Not a Time*, l'équivalent de `NaN` pour les durées)
dans les colonnes des séances qu'ils n'ont pas disputées.

In [6]:
afficher(session.results)

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps
16,16,C LECLERC,LEC,leclerc,Ferrari,E80020,ferrari,Charles,Leclerc,Charles Leclerc,https://media.formula1.com/d_driver_fallback_i...,MON,1.0,1.0,NaN,0 days 00:01:11.584000,0 days 00:01:10.825000,0 days 00:01:10.270000,NaT,,NaN,NaN
81,81,O PIASTRI,PIA,piastri,McLaren,FF8000,mclaren,Oscar,Piastri,Oscar Piastri,https://media.formula1.com/d_driver_fallback_i...,AUS,2.0,2.0,NaN,0 days 00:01:11.500000,0 days 00:01:10.756000,0 days 00:01:10.424000,NaT,,NaN,NaN
55,55,C SAINZ,SAI,sainz,Ferrari,E80020,ferrari,Carlos,Sainz,Carlos Sainz,https://media.formula1.com/d_driver_fallback_i...,ESP,3.0,3.0,NaN,0 days 00:01:11.543000,0 days 00:01:11.075000,0 days 00:01:10.518000,NaT,,NaN,NaN
4,4,L NORRIS,NOR,norris,McLaren,FF8000,mclaren,Lando,Norris,Lando Norris,https://media.formula1.com/d_driver_fallback_i...,GBR,4.0,4.0,NaN,0 days 00:01:11.760000,0 days 00:01:10.732000,0 days 00:01:10.542000,NaT,,NaN,NaN
63,63,G RUSSELL,RUS,russell,Mercedes,27F4D2,mercedes,George,Russell,George Russell,https://media.formula1.com/d_driver_fallback_i...,GBR,5.0,5.0,NaN,0 days 00:01:11.492000,0 days 00:01:10.929000,0 days 00:01:10.543000,NaT,,NaN,NaN
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671C6,red_bull,Max,Verstappen,Max Verstappen,https://media.formula1.com/d_driver_fallback_i...,NED,6.0,6.0,NaN,0 days 00:01:11.711000,0 days 00:01:10.745000,0 days 00:01:10.567000,NaT,,NaN,NaN
44,44,L HAMILTON,HAM,hamilton,Mercedes,27F4D2,mercedes,Lewis,Hamilton,Lewis Hamilton,https://media.formula1.com/d_driver_fallback_i...,GBR,7.0,7.0,NaN,0 days 00:01:11.528000,0 days 00:01:11.056000,0 days 00:01:10.621000,NaT,,NaN,NaN
22,22,Y TSUNODA,TSU,tsunoda,RB,6692FF,rb,Yuki,Tsunoda,Yuki Tsunoda,https://media.formula1.com/d_driver_fallback_i...,JPN,8.0,8.0,NaN,0 days 00:01:11.852000,0 days 00:01:11.106000,0 days 00:01:10.858000,NaT,,NaN,NaN
23,23,A ALBON,ALB,albon,Williams,64C4FF,williams,Alexander,Albon,Alexander Albon,https://media.formula1.com/d_driver_fallback_i...,THA,9.0,9.0,NaN,0 days 00:01:11.623000,0 days 00:01:11.216000,0 days 00:01:10.948000,NaT,,NaN,NaN
10,10,P GASLY,GAS,gasly,Alpine,0093cc,alpine,Pierre,Gasly,Pierre Gasly,https://media.formula1.com/d_driver_fallback_i...,FRA,10.0,10.0,NaN,0 days 00:01:11.714000,0 days 00:01:10.896000,0 days 00:01:11.311000,NaT,,NaN,NaN


✏️ **À toi :** parmi les colonnes ci-dessus, lesquelles forment un vrai classement de qualifications ?
Passe-les en liste à `afficher` (deuxième argument).

In [7]:
# ✏️ afficher(session.results, [...])
afficher(session.results, "Position")

16     1.0
81     2.0
55     3.0
4      4.0
63     5.0
1      6.0
44     7.0
22     8.0
23     9.0
10    10.0
31    11.0
3     12.0
18    13.0
27    14.0
14    14.0
2     15.0
20    15.0
11    16.0
77    17.0
24    18.0
Name: Position, dtype: float64

## Mission 4 ✏️ · Explorer `session.laps`

**Concept clé : une ligne = un tour d'un pilote.**
Chaque colonne décrit ce tour : son temps, le pneu utilisé, son numéro, les passages aux stands…

Avant toute analyse, on **regarde** les données. Sans ça, on risque de calculer des moyennes sur des
valeurs manquantes ou aberrantes sans s'en rendre compte.

Documentation utile : [Timing and Telemetry Data → Laps](https://docs.fastf1.dev/core.html#fastf1.core.Laps)
(chaque colonne y est décrite).

### 4.1 · La taille

Combien de lignes et de colonnes ? Est-ce cohérent avec le nombre de pilotes et de tours ?
Outil : l'attribut `.shape`.

In [8]:
# ✏️
afficher(session.laps.shape)

(426, 31)

### 4.2 · Les colonnes

Liste les colonnes, puis essaie de deviner ce que chacune représente avant de vérifier dans la documentation.
Outil : `.columns`.

In [9]:
# ✏️
afficher(session.laps.columns)

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')

### 4.3 · Les premières lignes

Utilise ta fonction `afficher` avec le paramètre `n`.

In [10]:
# ✏️
afficher(session.laps, None, 10)

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:17:37.749000,LEC,16,0 days 00:01:47.009000,1.0,1.0,0 days 00:15:51.382000,NaT,0 days 00:00:33.194000,0 days 00:00:44.490000,0 days 00:00:29.325000,0 days 00:16:24.018000,0 days 00:17:08.623000,0 days 00:17:37.812000,74.0,176.0,266.0,233.0,False,SOFT,1.0,True,Ferrari,0 days 00:15:51.382000,2024-05-25 14:02:56.065,1,NaN,False,,False,False
1,0 days 00:18:50.588000,LEC,16,0 days 00:01:12.839000,2.0,1.0,NaT,NaT,0 days 00:00:19.297000,0 days 00:00:34.137000,0 days 00:00:19.405000,0 days 00:17:57.046000,0 days 00:18:31.183000,0 days 00:18:50.588000,216.0,199.0,268.0,279.0,True,SOFT,2.0,True,Ferrari,0 days 00:17:37.749000,2024-05-25 14:04:42.432,1,NaN,False,,False,True
2,0 days 00:20:27.621000,LEC,16,0 days 00:01:37.033000,3.0,1.0,NaT,NaT,0 days 00:00:27.498000,0 days 00:00:43.667000,0 days 00:00:25.868000,0 days 00:19:18.086000,0 days 00:20:01.753000,0 days 00:20:27.621000,183.0,172.0,269.0,254.0,False,SOFT,3.0,True,Ferrari,0 days 00:18:50.588000,2024-05-25 14:05:55.271,1,NaN,False,,False,True
3,0 days 00:21:40.073000,LEC,16,0 days 00:01:12.452000,4.0,1.0,NaT,NaT,0 days 00:00:18.794000,0 days 00:00:34.388000,0 days 00:00:19.270000,0 days 00:20:46.415000,0 days 00:21:20.803000,0 days 00:21:40.073000,222.0,200.0,269.0,281.0,True,SOFT,4.0,True,Ferrari,0 days 00:20:27.621000,2024-05-25 14:07:32.304,1,NaN,False,,False,True
4,0 days 00:23:36.311000,LEC,16,0 days 00:01:56.238000,5.0,1.0,NaT,0 days 00:22:54.986000,0 days 00:00:21.593000,0 days 00:00:38.423000,0 days 00:00:56.222000,0 days 00:22:01.666000,0 days 00:22:40.089000,0 days 00:23:36.311000,196.0,175.0,NaN,257.0,False,SOFT,5.0,True,Ferrari,0 days 00:21:40.073000,2024-05-25 14:08:44.756,1,NaN,False,,False,False
5,0 days 00:25:18.839000,LEC,16,0 days 00:01:42.528000,6.0,2.0,0 days 00:23:37.442000,NaT,0 days 00:00:31.705000,0 days 00:00:42.437000,0 days 00:00:28.386000,0 days 00:24:08.016000,0 days 00:24:50.453000,0 days 00:25:18.839000,77.0,141.0,267.0,259.0,False,SOFT,6.0,False,Ferrari,0 days 00:23:36.311000,2024-05-25 14:10:40.994,1,NaN,False,,False,False
6,0 days 00:26:30.492000,LEC,16,0 days 00:01:11.653000,7.0,2.0,NaT,NaT,0 days 00:00:18.702000,0 days 00:00:33.831000,0 days 00:00:19.120000,0 days 00:25:37.541000,0 days 00:26:11.372000,0 days 00:26:30.492000,222.0,207.0,269.0,280.0,True,SOFT,7.0,False,Ferrari,0 days 00:25:18.839000,2024-05-25 14:12:23.522,12,NaN,False,,False,True
7,0 days 00:28:10.284000,LEC,16,0 days 00:01:39.792000,8.0,2.0,NaT,NaT,0 days 00:00:22.608000,0 days 00:00:40.264000,0 days 00:00:36.920000,0 days 00:26:53.100000,0 days 00:27:33.364000,0 days 00:28:10.284000,168.0,171.0,269.0,244.0,False,SOFT,8.0,False,Ferrari,0 days 00:26:30.492000,2024-05-25 14:13:35.175,1,NaN,False,,False,True
8,0 days 00:29:21.868000,LEC,16,0 days 00:01:11.584000,9.0,2.0,NaT,NaT,0 days 00:00:18.597000,0 days 00:00:33.825000,0 days 00:00:19.162000,0 days 00:28:28.881000,0 days 00:29:02.706000,0 days 00:29:21.868000,220.0,205.0,269.0,281.0,True,SOFT,9.0,False,Ferrari,0 days 00:28:10.284000,2024-05-25 14:15:14.967,1,NaN,False,,False,True
9,0 days 00:38:31.485000,LEC,16,NaT,10.0,2.0,NaT,0 days 00:30:51.141000,0 days 00:00:23.219000,0 days 00:00:45.996000,NaT,0 days 00:29:45.304000,0 days 00:30:31.230000,NaT,71.0,162.0,NaN,245.0,False,SOFT,10.0,False,Ferrari,0 days 00:29:21.868000,2024-05-25 14:16:26.551,12,NaN,False,,False,False


### 4.4 · Les types de données

Repère au moins trois types différents (`object`, `float64`, `timedelta64`…).
Pourquoi `LapTime` n'est-il pas un simple nombre ?
Outils : `.dtypes` ou `.info()`.

In [11]:
# ✏️
afficher(session.laps.dtypes)

Time                  timedelta64[ns]
Driver                         object
DriverNumber                   object
LapTime               timedelta64[ns]
LapNumber                     float64
Stint                         float64
PitOutTime            timedelta64[ns]
PitInTime             timedelta64[ns]
Sector1Time           timedelta64[ns]
Sector2Time           timedelta64[ns]
Sector3Time           timedelta64[ns]
Sector1SessionTime    timedelta64[ns]
Sector2SessionTime    timedelta64[ns]
Sector3SessionTime    timedelta64[ns]
SpeedI1                       float64
SpeedI2                       float64
SpeedFL                       float64
SpeedST                       float64
IsPersonalBest                   bool
Compound                       object
TyreLife                      float64
FreshTyre                        bool
Team                           object
LapStartTime          timedelta64[ns]
LapStartDate           datetime64[ns]
TrackStatus                    object
Position    

### 4.5 · Les valeurs manquantes

Combien de valeurs manquent dans chaque colonne ? Surtout, cherche **pourquoi**
(premier tour ? tour de sortie des stands ? drapeau rouge ?).
Outil : `.isna().sum()`.

In [12]:
# ✏️
afficher(session.laps.isna().sum())

Time                    0
Driver                  0
DriverNumber            0
LapTime                79
LapNumber               0
Stint                   0
PitOutTime            334
PitInTime             334
Sector1Time             2
Sector2Time             1
Sector3Time            78
Sector1SessionTime      2
Sector2SessionTime      1
Sector3SessionTime     78
SpeedI1                 0
SpeedI2                 0
SpeedFL                92
SpeedST                 0
IsPersonalBest          0
Compound                0
TyreLife                0
FreshTyre               0
Team                    0
LapStartTime            0
LapStartDate            0
TrackStatus             0
Position              426
Deleted                 0
DeletedReason           0
FastF1Generated         0
IsAccurate              0
dtype: int64

### 4.6 · Les colonnes importantes pour la suite

Observe en particulier `Driver`, `LapNumber`, `LapTime`, `Compound`, `TyreLife`,
`PitInTime`, `PitOutTime` et `IsAccurate`. Quelles valeurs prennent-elles ?
Outils : `.unique()`, `.value_counts()`.

In [13]:
# ✏️
afficher(session.laps["Driver"].value_counts())

Driver
GAS    30
ALB    29
SAI    28
RUS    28
HAM    28
VER    28
NOR    27
LEC    26
TSU    25
PIA    24
RIC    21
OCO    20
HUL    20
MAG    18
STR    17
SAR    12
PER    12
ALO    11
BOT    11
ZHO    11
Name: count, dtype: int64

## ✏️ Bilan de l'exploration

*Résume ici en quelques lignes ce que contiennent les données et les pièges repérés
(valeurs manquantes, types particuliers, tours à exclure…). Ce résumé servira pour le README.*

- …

In [14]:
la session contient un grand nombre de donnée. Il a plusieurs attributs comme les resultats de la séance que tu as choisi durant le load de la session et aussi 

SyntaxError: invalid syntax (1176866201.py, line 1)